# 03 — Data Preprocessing

## Project: Movie Recommendation System

### Objective

The objective of this notebook is to clean and prepare the TMDB 5000 Movies and Credits datasets for the recommendation system.

The raw datasets contain missing values, duplicate records, and structured information stored as JSON-like strings.

We will process these datasets and extract useful movie information required for content-based recommendation.

### Main Tasks

1. Load the original datasets.
2. Merge movies and credits.
3. Select relevant features.
4. Handle missing values.
5. Remove duplicate records.
6. Parse genres, keywords, cast, and crew.
7. Extract directors and actors.
8. Save the cleaned dataset.

### Expected Outcome

A cleaned and structured movie dataset ready for feature engineering.

## 1. Import Useful Libraries

In [22]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## 2. Define Project Paths

We define paths to the raw datasets and the directory where the processed dataset will be saved.

The original datasets will remain unchanged. All preprocessing operations will be performed on DataFrame copies.

In [23]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

MOVIES_PATH = RAW_DATA_DIR / "tmdb_5000_movies.csv"
CREDITS_PATH = RAW_DATA_DIR / "tmdb_5000_credits.csv"

OUTPUT_PATH = INTERIM_DATA_DIR / "movies_cleaned.csv"

assert MOVIES_PATH.exists(), f"Missing file: {MOVIES_PATH}"
assert CREDITS_PATH.exists(), f"Missing file: {CREDITS_PATH}"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully.")

Project paths configured successfully.


## 5. Load Datasets

Load the original TMDB Movies and Credits CSV files.

We will keep the raw datasets separate until we understand their relationship and merge them.

In [24]:
movies_df = pd.read_csv(MOVIES_PATH)
credits_df = pd.read_csv(CREDITS_PATH)

print("Movies dataset:", movies_df.shape)
print("Credits dataset:", credits_df.shape)

Movies dataset: (4803, 20)
Credits dataset: (4803, 4)


## 6. Merge Movies and Credits

The two datasets contain complementary information.

The Movies dataset contains movie information such as:

- Movie ID
- Title
- Overview
- Genres
- Keywords
- Popularity
- Ratings

The Credits dataset contains:

- Movie ID
- Cast
- Crew

We use the movie identifiers to combine both datasets into a single DataFrame.

The relationship is:

- Movies: `id`
- Credits: `movie_id`

In [25]:
print("Movies columns:")
print(movies_df.columns.tolist())

print("\nCredits columns:")
print(credits_df.columns.tolist())

Movies columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

Credits columns:
['movie_id', 'title', 'cast', 'crew']


In [26]:
assert "id" in movies_df.columns
assert "movie_id" in credits_df.columns

print("Duplicate movie IDs in movies:", movies_df["id"].duplicated().sum())
print("Duplicate movie IDs in credits:", credits_df["movie_id"].duplicated().sum())

Duplicate movie IDs in movies: 0
Duplicate movie IDs in credits: 0


In [27]:
# Merge movie metadata with cast/crew credits into a single DataFrame
merged_df = movies_df.merge(
    credits_df,
    left_on="id",  # Join key in movies_df
    right_on="movie_id",  # Corresponding join key in credits_df
    how="inner",  # Keep only records present in both DataFrames
    suffixes=(
        "_movie",
        "_credits",
    ),  # Disambiguate overlapping column names (e.g., 'title')
    validate="one_to_one",  # Ensure join keys are unique in both tables to prevent row duplication
)


print("Merged dataset shape:", merged_df.shape)

display(merged_df.head())

Merged dataset shape: (4803, 24)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title_movie,vote_average,vote_count,movie_id,title_credits,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""sp...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, ...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century Fox Film Corporatio...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""GB"", ""name"": ""United ...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a3685532001c9a"", ""gender"": ...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"": ""Editing"", ""gender"": 0, ""id"": 1721, ""jo..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic is...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of t...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Films"", ""id"": 130}, {""na...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""credit_id"": ""52fe4232c3a36847f800b50d"", ""g...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"": ""Camera"", ""gender"": 2, ""id"": 120, ""job""..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret...",en,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. Whil...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""name"": ""Danjaq"", ""id"": 10761}, {""name"": ""B24"", ""id"": ...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name"": ""United States of ...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""}, {""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""credit_id"": ""52fe4d22c3a368484e1d8d6b"", ""gender"": 2,...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""department"": ""Sound"", ""gender"": 2, ""id"": 153, ""job"":..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""name"": ""Crime""}, {""id"": 18, ""name"": ""Drama""}, {""id"": ...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853, ""name"": ""crime fighter""}, {""id"": 949, ""name"": ""te...",en,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's c...",112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""

## 7. Select Relevant Columns

Not every column in the original datasets is required for our content-based recommendation system.

We select the information that describes the content of a movie.

Selected features:

- id: Unique movie identifier.
- title: Movie title.
- overview: Movie description.
- genres: Movie genres.
- keywords: Movie-related keywords.
- cast: Actors appearing in the movie.
- crew: People involved in production.

These features will be processed and used during feature engineering.

In [28]:
required_columns = [
    "id",
    "title_movie",
    "overview",
    "genres",
    "keywords",
    "cast",
    "crew",
]

missing_columns = [
    column for column in required_columns if column not in merged_df.columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

movies_cleaned = merged_df[required_columns].copy()

movies_cleaned = movies_cleaned.rename(columns={"title_movie": "title"})

display(movies_cleaned.head())

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, ...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""sp...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a3685532001c9a"", ""gender"": ...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"": ""Editing"", ""gender"": 0, ""id"": 1721, ""jo..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of t...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic is...","[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""credit_id"": ""52fe4232c3a36847f800b50d"", ""g...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"": ""Camera"", ""gender"": 2, ""id"": 120, ""job""..."
2,206647,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. Whil...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret...","[{""cast_id"": 1, ""character"": ""James Bond"", ""credit_id"": ""52fe4d22c3a368484e1d8d6b"", ""gender"": 2,...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""department"": ""Sound"", ""gender"": 2, ""id"": 153, ""job"":..."
3,49026,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's c...","[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""name"": ""Crime""}, {""id"": 18, ""name"": ""Drama""}, {""id"": ...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853, ""name"": ""crime fighter""}, {""id"": 949, ""name"": ""te...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Batman"", ""credit_id"": ""52fe4781c3a36847f8139869"", ""g...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""department"": ""Sound"", ""gender"": 2, ""id"": 947, ""job"":..."
4,49529,John Carter,"John Carter is a war-weary, former military captain who's inexplicably transported to the myster...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 878, ""name"": ""Science Fic...","[{""id"": 818, ""name"": ""based on novel""}, {""id"": 839, ""name"": ""mars""}, {""id"": 1456, ""name"": ""medal...","[{""cast_id"": 5, ""character"": ""John Carter"", ""credit_id"": ""52fe479ac3a36847f813ea75"", ""gender"": 2...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""department"": ""Writing"", ""gender"": 2, ""id"": 7, ""job"":..."


## 8. Handle Missing Values

Missing values can cause problems during text processing and recommendation generation.

We inspect missing values in the selected columns before deciding how to handle them.

For this project:

- Missing titles will be removed because a movie needs a valid title.
- Missing overviews will be replaced with empty strings.
- Missing genres, keywords, cast, and crew will be replaced with empty lists.

We do not invent information that is unavailable in the original dataset.

In [29]:
missing_values = movies_cleaned.isnull().sum()

display(missing_values[missing_values > 0].to_frame("Missing Values"))

,Missing Values
overview,3


In [30]:
# Remove records without a movie title.
movies_cleaned = movies_cleaned.dropna(subset=["title"])

# Replace missing textual descriptions.
movies_cleaned["overview"] = movies_cleaned["overview"].fillna("")

# Replace missing structured fields.
structured_columns = ["genres", "keywords", "cast", "crew"]

for column in structured_columns:
    movies_cleaned[column] = movies_cleaned[column].fillna("[]")

print("Missing values after handling:")
display(movies_cleaned.isnull().sum().to_frame("Missing Values"))

Missing values after handling:


,Missing Values
id,0
title,0
overview,0
genres,0
keywords,0
cast,0
crew,0


## 9. Remove Duplicate Records

Duplicate movie records can affect the recommendation system and create inconsistent results.

We check for duplicate movie identifiers and remove duplicate records if they exist.

The movie ID is treated as the unique identifier.

In [31]:
print("Duplicate movie IDs before removal:", movies_cleaned["id"].duplicated().sum())

movies_cleaned = movies_cleaned.drop_duplicates(subset=["id"], keep="first").copy()

print("Duplicate movie IDs after removal:", movies_cleaned["id"].duplicated().sum())

print("Dataset shape:", movies_cleaned.shape)

Duplicate movie IDs before removal: 0
Duplicate movie IDs after removal: 0
Dataset shape: (4803, 7)


## 10. Parse Structured Columns

Some TMDB columns store structured information as strings representing Python lists of dictionaries.

For example, the genres column may contain:

[
    {"id": 28, "name": "Action"},
    {"id": 12, "name": "Adventure"}
]

These values need to be parsed before we can extract useful information.

We use ast.literal_eval() to safely evaluate Python literal structures rather than using eval().

In [32]:
def parse_data(value):
    """Convert a string representation of a list into a Python list.

    Args:
        value: The input to parse, typically a stringified list (e.g., "[1, 2]")
               or an already-parsed list.

    Returns:
        The parsed list, or an empty list if the input is missing, invalid,
        or does not evaluate to a list.
    """
    # Return immediately if the input is already a list
    if isinstance(value, list):
        return value

    # Guard against non-strings (like None, int, dict) or empty/whitespace-only strings
    if not isinstance(value, str) or not value.strip():
        return []

    try:
        # Safely evaluate the string literal without executing arbitrary code
        parsed_value = ast.literal_eval(value)

        # Ensure the evaluated result is specifically a list (not a tuple, dict, etc.)
        if isinstance(parsed_value, list):
            return parsed_value
    except (ValueError, SyntaxError, TypeError):
        # Ignore malformed strings or syntax errors and fall back to an empty list
        pass

    return []

In [33]:
for column in structured_columns:
    movies_cleaned[column] = movies_cleaned[column].apply(parse_data)

display(movies_cleaned[["title", "genres", "keywords", "cast", "crew"]].head())

,title,genres,keywords,cast,crew
0,Avatar,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {...","[{'id': 1463, 'name': 'culture clash'}, {'id': 2964, 'name': 'future'}, {'id': 3386, 'name': 'sp...","[{'cast_id': 242, 'character': 'Jake Sully', 'credit_id': '5602a8a7c3a3685532001c9a', 'gender': ...","[{'credit_id': '52fe48009251416c750aca23', 'department': 'Editing', 'gender': 0, 'id': 1721, 'jo..."
1,Pirates of the Caribbean: At World's End,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 28, 'name': 'Action'}]","[{'id': 270, 'name': 'ocean'}, {'id': 726, 'name': 'drug abuse'}, {'id': 911, 'name': 'exotic is...","[{'cast_id': 4, 'character': 'Captain Jack Sparrow', 'credit_id': '52fe4232c3a36847f800b50d', 'g...","[{'credit_id': '52fe4232c3a36847f800b579', 'department': 'Camera', 'gender': 2, 'id': 120, 'job'..."
2,Spectre,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 80, 'name': 'Crime'}]","[{'id': 470, 'name': 'spy'}, {'id': 818, 'name': 'based on novel'}, {'id': 4289, 'name': 'secret...","[{'cast_id': 1, 'character': 'James Bond', 'credit_id': '52fe4d22c3a368484e1d8d6b', 'gender': 2,...","[{'credit_id': '54805967c3a36829b5002c41', 'department': 'Sound', 'gender': 2, 'id': 153, 'job':..."
3,The Dark Knight Rises,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'name': 'Crime'}, {'id': 18, 'name': 'Drama'}, {'id': ...","[{'id': 849, 'name': 'dc comics'}, {'id': 853, 'name': 'crime fighter'}, {'id': 949, 'name': 'te...","[{'cast_id': 2, 'character': 'Bruce Wayne / Batman', 'credit_id': '52fe4781c3a36847f8139869', 'g...","[{'credit_id': '52fe4781c3a36847f81398c3', 'department': 'Sound', 'gender': 2, 'id': 947, 'job':..."
4,John Carter,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 878, 'name': 'Science Fic...","[{'id': 818, 'name': 'based on novel'}, {'id': 839, 'name': 'mars'}, {'id': 1456, 'name': 'medal...","[{'cast_id': 5, 'character': 'John Carter', 'credit_id': '52fe479ac3a36847f813ea75', 'gender': 2...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'department': 'Writing', 'gender': 2, 'id': 7, 'job':..."


## 11. Extract Genres and Keywords

The genres and keywords columns contain lists of dictionaries.

For the recommendation system, we need the names rather than the complete dictionary structures.

We extract the names from these columns.

Example:

Input:
[
    {"id": 28, "name": "Action"},
    {"id": 12, "name": "Adventure"}
]

Output:
["Action", "Adventure"]

In [34]:
# def extract_names(items):
#     """Extract the name field from a list of dictionaries."""

#     return [
#         item["name"]
#         for item in items
#         if isinstance(item, dict)
#         and isinstance(item.get("name"), str)
#         and item["name"].strip()
#     ]

In [35]:
def extract_names(items):
    """Extract non-empty name strings from a list of dictionaries."""
    names = []

    for item in items:
        # Skip if item is not a dictionary
        if not isinstance(item, dict):
            continue

        name = item.get("name")

        # Add name only if it's a non-empty string
        if isinstance(name, str) and name.strip():
            names.append(name)

    return names

In [36]:
movies_cleaned["genres"] = movies_cleaned["genres"].apply(extract_names)

movies_cleaned["keywords"] = movies_cleaned["keywords"].apply(extract_names)

display(movies_cleaned[["title", "genres", "keywords"]].head(10))

,title,genres,keywords
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, spa..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, ship..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]"
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham..."
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edg..."
5,Spider-Man 3,"[Fantasy, Action, Adventure]","[dual identity, amnesia, sandstorm, love of one's life, forgiveness, spider, wretch, death of a ..."
6,Tangled,"[Animation, Family]","[hostage, magic, horse, fairy tale, musical, princess, animation, tower, blonde woman, selfishne..."
7,Avengers: Age of Ultron,"[Action, Adventure, Science Fiction]","[marvel comic, sequel, superhero, based on comic book, vision, superhero team, duringcreditsstin..."
8,Harry Potter and the Half-Blood Prince,"[Adventure, Fantasy, Family]","[witch, magic, broom, school of witchcraft, wizardry, apparition, teenage crush, werewolf]"
9,Batman v Superman: Dawn of Justice,"[Action, Adventure, Fantasy]","[dc comics, vigilante, superhero, based on comic book, revenge, super powers, clark kent, bruce ..."


## 12. Extract Cast Members

The cast column contains information about actors appearing in each movie.

For this project, we extract the first three cast members from each movie.

The first few cast members can provide useful information about the movie's leading actors.

We retain the actor names rather than the complete cast dictionaries.

In [37]:
# def extract_cast(cast_list, limit=3):
#     """Extract the names of the first few cast members."""

#     return [
#         member["name"]
#         for member in cast_list[:limit]
#         if isinstance(member, dict)
#         and isinstance(member.get("name"), str)
#         and member["name"].strip()
#     ]

In [38]:
def extract_cast(cast_list, limit=3):
    """Extract the names of the first few cast members."""
    names = []

    for member in cast_list:
        # Stop once we have collected enough valid names
        if len(names) == limit:
            break

        # Skip if member is not a dictionary
        if not isinstance(member, dict):
            continue

        name = member.get("name")

        # Add name only if it's a non-empty string
        if isinstance(name, str) and name.strip():
            names.append(name)

    return names

In [39]:
movies_cleaned["cast"] = movies_cleaned["cast"].apply(extract_cast)

display(movies_cleaned[["title", "cast"]].head(10))

,title,cast
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]"
3,The Dark Knight Rises,"[Christian Bale, Michael Caine, Gary Oldman]"
4,John Carter,"[Taylor Kitsch, Lynn Collins, Samantha Morton]"
5,Spider-Man 3,"[Tobey Maguire, Kirsten Dunst, James Franco]"
6,Tangled,"[Zachary Levi, Mandy Moore, Donna Murphy]"
7,Avengers: Age of Ultron,"[Robert Downey Jr., Chris Hemsworth, Mark Ruffalo]"
8,Harry Potter and the Half-Blood Prince,"[Daniel Radcliffe, Rupert Grint, Emma Watson]"
9,Batman v Superman: Dawn of Justice,"[Ben Affleck, Henry Cavill, Gal Gadot]"


## 13. Extract Director

The crew column contains information about people involved in movie production.

We extract the director's name using the following conditions:

- The department should be Directing.
- The job should be Director.

Some movies may not have a director entry in the available dataset. In those cases, we return an empty list.

In [40]:
# def extract_director(crew_list):
#     """Extract the director's name from the crew list."""

#     directors = [
#         member["name"]
#         for member in crew_list
#         if isinstance(member, dict)
#         and member.get("job") == "Director"
#         and isinstance(member.get("name"), str)
#         and member["name"].strip()
#     ]

#     return directors[:1]

In [41]:
def extract_director(crew_list):
    """Extract the director's name from the crew list."""
    for member in crew_list:
        # Skip if member is not a dictionary
        if not isinstance(member, dict):
            continue

        # Check if this crew member is the Director
        if member.get("job") == "Director":
            name = member.get("name")

            # Return a single-item list as soon as a valid name is found
            if isinstance(name, str) and name.strip():
                return [name]

    # Return an empty list if no valid director was found
    return []

In [42]:
movies_cleaned["crew"] = movies_cleaned["crew"].apply(extract_director)

movies_cleaned = movies_cleaned.rename(columns={"crew": "director"})

display(movies_cleaned[["title", "director"]].head(10))

,title,director
0,Avatar,[James Cameron]
1,Pirates of the Caribbean: At World's End,[Gore Verbinski]
2,Spectre,[Sam Mendes]
3,The Dark Knight Rises,[Christopher Nolan]
4,John Carter,[Andrew Stanton]
5,Spider-Man 3,[Sam Raimi]
6,Tangled,[Byron Howard]
7,Avengers: Age of Ultron,[Joss Whedon]
8,Harry Potter and the Half-Blood Prince,[David Yates]
9,Batman v Superman: Dawn of Justice,[Zack Snyder]


## 14. Final Dataset Inspection

After completing preprocessing, we inspect the resulting dataset.

We verify:

- Dataset dimensions.
- Column names.
- Data types.
- Missing values.
- Duplicate movie IDs.
- Sample processed records.

This ensures that the dataset is ready for the next stage.

In [43]:
print("Final dataset shape:", movies_cleaned.shape)

print("\nFinal columns:")
print(movies_cleaned.columns.tolist())

print("\nData types:")
print(movies_cleaned.dtypes)

print("\nMissing values:")
display(movies_cleaned.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate IDs:")
print(movies_cleaned["id"].duplicated().sum())

Final dataset shape: (4803, 7)

Final columns:
['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'director']

Data types:
id           int64
title          str
overview       str
genres      object
keywords    object
cast        object
director    object
dtype: object

Missing values:


,Missing Values
id,0
title,0
overview,0
genres,0
keywords,0
cast,0
director,0



Duplicate IDs:
0


In [44]:
display(movies_cleaned.head(10))

,id,title,overview,genres,keywords,cast,director
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, ...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, spa...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of t...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, ship...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. Whil...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's c...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military captain who's inexplicably transported to the myster...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edg...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]
5,559,Spider-Man 3,The seemingly invincible Spider-Man goes up against an all-new crop of villain – including the s...,"[Fantasy, Action, Adventure]","[dual identity, amnesia, sandstorm, love of one's life, forgiveness, spider, wretch, death of a ...","[Tobey Maguire, Kirsten Dunst, James Franco]",[Sam Raimi]
6,38757,Tangled,When the kingdom's most wanted-and most charming-bandit Flynn Rider hides out in a mysterious to...,"[Animation, Family]","[hostage, magic, horse, fairy tale, musical, princess, animation, tower, blonde woman, selfishne...","[Zachary Levi, Mandy Moore, Donna Murphy]",[Byron Howard]
7,99861,Avengers: Age of Ultron,"When Tony Stark tries to jumpstart a dormant peacekeeping program, things go awry and Earth’s Mi...","[Action, Adventure, Science Fiction]","[marvel comic, sequel, superhero, based on comic book, vision, superhero team, duringcreditsstin...","[Robert Downey Jr., Chris Hemsworth, Mark Ruffalo]",[Joss Whedon]
8,767,Harry Potter and the Half-Blood Prince,"As Harry begins his sixth year at Hogwarts, he discovers an old book marked as 'Property of the ...","[Adventure, Fantasy, Family]","[witch, magic, broom, school of witchcraft, wizardry, apparition, teenage crush, werewolf]","[Daniel Radcliffe, Rupert Grint, Emma Watson]",[David Yates]
9,209112,Batman v Superman: Dawn of Justice,"Fearing the actions of a god-like Super Hero left unchecked, Gotham City’s own formidable, force...","[Action, Adventure, Fantasy]","[dc comics, vigilante, superhero, based on comic book, revenge, super powers, clark kent, bruce ...","[Ben Affleck, Henry Cavill, Gal Gadot]",[Zack Snyder]


## 15. Save Preprocessed Dataset

The cleaned dataset is saved in the data/interim directory.

This keeps the original raw datasets unchanged and makes the preprocessing output available for subsequent notebooks.

The saved dataset will be used in the feature engineering stage.

In [45]:
movies_cleaned.to_csv(OUTPUT_PATH, index=False)

print(f"Preprocessed dataset saved to: {OUTPUT_PATH}")
print(f"Total movies saved: {len(movies_cleaned)}")

Preprocessed dataset saved to: d:\Learning-Materials\Projects\ML-Projects\Content-Based-Movie-Recommendation-System\data\interim\movies_cleaned.csv
Total movies saved: 4803


## 16. Conclusion

In this notebook, we performed preprocessing on the TMDB 5000 Movies and Credits datasets.

### Tasks Completed

- Loaded the original datasets.
- Merged movies and credits using movie identifiers.
- Selected relevant movie features.
- Handled missing values.
- Removed duplicate movie records.
- Parsed structured JSON-like columns.
- Extracted genres and keywords.
- Extracted the first three cast members.
- Extracted movie directors.
- Saved the cleaned dataset.

### Output

The cleaned dataset is stored at:

data/interim/movies_cleaned.csv

### Next Steps

In the next notebook, we will perform Feature Engineering.

We will prepare the extracted movie information for content-based recommendation by transforming the selected features into a suitable representation.

**Next Notebook:** 04_feature_engineering.ipynb